<a href="https://colab.research.google.com/github/soumyanildey/Machine-Learning-Projects/blob/main/MedBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install -U requests faiss-cpu sentence-transformers langchain tqdm BitsandBytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 73.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12


In [23]:
medical_situations = [
    # Cardiovascular Conditions
    "heart attack", "stroke", "hypertension", "arrhythmia", "heart failure",
    "coronary artery disease", "cardiac arrest", "deep vein thrombosis",
    "pulmonary embolism", "aortic aneurysm", "peripheral artery disease",

    # Neurological Conditions
    "epilepsy", "migraine", "Alzheimer's disease", "Parkinson's disease",
    "multiple sclerosis", "meningitis", "encephalitis", "brain tumor",
    "cerebral palsy", "spinal cord injury",

    # Respiratory Conditions
    "pneumonia", "asthma", "chronic obstructive pulmonary disease (COPD)",
    "tuberculosis", "lung cancer", "bronchitis", "sleep apnea",
    "pulmonary fibrosis", "cystic fibrosis",

    # Endocrine & Metabolic Disorders
    "diabetes", "hyperthyroidism", "hypothyroidism", "Cushing's syndrome",
    "Addison's disease", "pituitary disorders", "osteoporosis",

    # Gastrointestinal Conditions
    "gastritis", "ulcer", "gastroesophageal reflux disease (GERD)",
    "irritable bowel syndrome (IBS)", "Crohn's disease", "ulcerative colitis",
    "hepatitis", "cirrhosis", "pancreatitis", "colon cancer",

    # Renal & Urinary Conditions
    "kidney stones", "chronic kidney disease", "urinary tract infection (UTI)",
    "bladder cancer", "nephritis", "prostate cancer", "acute kidney injury",

    # Hematologic & Immune System Conditions
    "anemia", "leukemia", "lymphoma", "sickle cell disease",
    "HIV/AIDS", "systemic lupus erythematosus (SLE)", "rheumatoid arthritis",
    "autoimmune disorders",

    # Infections & Communicable Diseases
    "COVID-19", "influenza", "dengue fever", "malaria", "Zika virus",
    "Ebola", "chickenpox", "measles", "mumps", "rubella",
    "tetanus", "rabies", "sepsis",

    # Dermatological Conditions
    "psoriasis", "eczema", "melanoma", "skin infections",
    "acne", "alopecia", "dermatitis",

    # Musculoskeletal Disorders
    "arthritis", "osteoporosis", "fracture", "tendinitis",
    "fibromyalgia", "carpal tunnel syndrome", "gout",

    # Mental Health Disorders
    "depression", "anxiety", "bipolar disorder", "schizophrenia",
    "obsessive-compulsive disorder (OCD)", "post-traumatic stress disorder (PTSD)",
    "eating disorders", "substance abuse",

    # Gynecological & Obstetric Conditions
    "pregnancy complications", "ectopic pregnancy", "preeclampsia",
    "gestational diabetes", "ovarian cysts", "endometriosis",
    "polycystic ovary syndrome (PCOS)", "cervical cancer",
    "menopause complications",

    # Pediatric Conditions
    "congenital heart defects", "cystic fibrosis", "Down syndrome",
    "neonatal jaundice", "cerebral palsy", "autism spectrum disorder",
    "whooping cough (pertussis)", "pediatric asthma",

    # Ophthalmological & ENT Conditions
    "glaucoma", "cataracts", "macular degeneration",
    "retinal detachment", "conjunctivitis", "hearing loss",
    "otitis media", "sinusitis", "tinnitus",

    # Toxicological & Emergency Conditions
    "poisoning", "drug overdose", "burns", "electric shock",
    "anaphylaxis", "heat stroke", "frostbite",
    "drowning", "snake bite", "allergic reaction",

    # Others
    "cancer", "tumors", "rare genetic disorders"
]

medical_situations.extend([
    # Infectious Diseases
    "Tuberculosis", "Influenza", "HIV/AIDS", "Malaria", "Dengue Fever",
    "Hepatitis B", "Hepatitis C", "COVID-19", "Ebola", "Zika Virus",
    "Cholera", "Typhoid Fever", "Measles", "Mumps", "Rubella",
    "Tetanus", "Rabies", "Leprosy", "Lyme Disease", "Syphilis",

    # Genetic Disorders
    "Cystic Fibrosis", "Sickle Cell Anemia", "Huntington’s Disease",
    "Duchenne Muscular Dystrophy", "Marfan Syndrome", "Tay-Sachs Disease",
    "Phenylketonuria (PKU)", "Hemophilia", "Fragile X Syndrome",
    "Ehlers-Danlos Syndrome", "Albinism", "Wilson’s Disease",

    # Autoimmune Diseases
    "Rheumatoid Arthritis", "Multiple Sclerosis", "Lupus (SLE)",
    "Type 1 Diabetes", "Graves’ Disease", "Hashimoto’s Thyroiditis",
    "Celiac Disease", "Psoriasis", "Sjogren’s Syndrome",
    "Ankylosing Spondylitis", "Myasthenia Gravis",

    # Neurological Disorders
    "Alzheimer’s Disease", "Parkinson’s Disease", "Epilepsy",
    "Amyotrophic Lateral Sclerosis (ALS)", "Multiple Sclerosis",
    "Huntington’s Disease", "Migraine", "Tourette Syndrome",
    "Guillain-Barré Syndrome", "Narcolepsy",

    # Cardiovascular Diseases
    "Hypertension", "Coronary Artery Disease", "Heart Failure",
    "Arrhythmia", "Stroke", "Aortic Aneurysm", "Peripheral Artery Disease",
    "Congenital Heart Defects", "Cardiomyopathy",

    # Respiratory Diseases
    "Asthma", "Chronic Obstructive Pulmonary Disease (COPD)",
    "Pulmonary Fibrosis", "Pneumonia", "Bronchitis", "Cystic Fibrosis",

    # Bone & Joint Disorders
    "Osteoporosis", "Osteoarthritis", "Rheumatoid Arthritis",
    "Gout", "Paget’s Disease of Bone", "Fibrodysplasia Ossificans Progressiva (FOP)",
    "Osteogenesis Imperfecta", "Osteopetrosis",

    # Endocrine Disorders
    "Diabetes Mellitus (Type 1 & 2)", "Cushing’s Syndrome",
    "Addison’s Disease", "Hyperthyroidism", "Hypothyroidism",
    "Polycystic Ovary Syndrome (PCOS)", "Acromegaly",

    # Gastrointestinal Diseases
    "Crohn’s Disease", "Ulcerative Colitis", "Irritable Bowel Syndrome (IBS)",
    "Gastroesophageal Reflux Disease (GERD)", "Celiac Disease",
    "Liver Cirrhosis", "Pancreatitis",

    # Skin Disorders
    "Eczema", "Psoriasis", "Vitiligo", "Melanoma",
    "Acne", "Rosacea", "Alopecia Areata",

    # Rare Diseases
    "Progeria", "Metachromatic Leukodystrophy (MLD)", "FOP",
    "Kawasaki Disease", "Rett Syndrome", "Wilson’s Disease",
    "Batten Disease", "Canavan Disease",

    # Cancers
    "Lung Cancer", "Breast Cancer", "Leukemia",
    "Lymphoma", "Colorectal Cancer", "Pancreatic Cancer",
    "Prostate Cancer", "Brain Tumor", "Melanoma",

    # Mental Health Disorders
    "Depression", "Anxiety Disorders", "Bipolar Disorder",
    "Schizophrenia", "Obsessive-Compulsive Disorder (OCD)",
    "Post-Traumatic Stress Disorder (PTSD)", "Autism Spectrum Disorder",

    # Blood Disorders
    "Anemia", "Thalassemia", "Hemophilia", "Leukopenia",
    "Polycythemia Vera", "Disseminated Intravascular Coagulation (DIC)",

    # Kidney & Urinary Diseases
    "Chronic Kidney Disease (CKD)", "Nephrotic Syndrome",
    "Polycystic Kidney Disease", "Glomerulonephritis", "Urinary Tract Infection (UTI)",

    # Eye Disorders
    "Glaucoma", "Cataracts", "Macular Degeneration",
    "Diabetic Retinopathy", "Retinitis Pigmentosa", "Keratoconus"
])


medical_situations.extend([
    "Abdominal aortic aneurysm",
    "Achilles tendinopathy",
    "Acne",
    "Acute cholecystitis",
    "Acute lymphoblastic leukaemia",
    "Acute lymphoblastic leukaemia: Children",
    "Acute lymphoblastic leukaemia: Teenagers and young adults",
    "Acute myeloid leukaemia",
    "Acute myeloid leukaemia: Children",
    "Acute myeloid leukaemia: Teenagers and young adults",
    "Acute pancreatitis",
    "Addison’s disease",
    "Adenomyosis",
    "Alcohol-related liver disease",
    "Allergic rhinitis",
    "Allergies",
    "Alzheimer’s disease",
    "Anal cancer",
    "Anaphylaxis",
    "Angina",
    "Angioedema",
    "Ankle sprain",
    "Ankle avulsion fracture",
    "Ankylosing spondylitis",
    "Anorexia nervosa",
    "Anxiety",
    "Anxiety disorders in children",
    "Appendicitis",
    "Arterial thrombosis",
    "Arthritis",
    "Asbestosis",
    "Asthma",
    "Ataxia",
    "Atopic eczema",
    "Atrial fibrillation",
    "Attention deficit hyperactivity disorder (ADHD)",
    "Autism",
    "Back problems",
    "Bacterial vaginosis",
    "Benign prostate enlargement",
    "Bile duct cancer (cholangiocarcinoma)",
    "Binge eating",
    "Bipolar disorder",
    "Bladder cancer",
    "Blood poisoning (sepsis)",
    "Bone cancer",
    "Bone cancer: Teenagers and young adults",
    "Bowel cancer",
    "Bowel incontinence",
    "Bowel polyps",
    "Brain stem death",
    "Brain tumours",
    "Brain tumours: Children",
    "Brain tumours: Teenagers and young adults",
    "Breast cancer (female)",
    "Breast cancer (male)",
    "Bronchiectasis",
    "Bronchitis",
    "Bulimia nervosa",
    "Bunion",
    "Carcinoid syndrome and carcinoid tumours",
    "Cardiovascular disease",
    "Carpal tunnel syndrome",
    "Catarrh",
    "Cellulitis",
    "Cerebral palsy",
    "Cervical cancer",
    "Cervical spondylosis",
    "Chest and rib injury",
    "Chest infection",
    "Chickenpox",
    "Chilblains",
    "Chlamydia",
    "Chronic fatigue syndrome",
    "Chronic kidney disease",
    "Chronic lymphocytic leukaemia",
    "Chronic myeloid leukaemia",
    "Chronic obstructive pulmonary disease (COPD)",
    "Chronic pain",
    "Chronic pancreatitis",
    "Cirrhosis",
    "Clavicle (collar bone) fracture",
    "Clostridium difficile",
    "Coeliac disease",
    "Cold sore",
    "Coma",
    "Common cold",
    "Concussion",
    "Congenital heart disease",
    "Conjunctivitis",
    "Constipation",
    "Coronary heart disease",
    "Coronavirus (COVID-19)",
    "Coronavirus (COVID-19): Longer-term effects (long COVID)",
    "Costochondritis",
    "Cough",
    "Crohn’s disease",
    "Croup",
    "Cystic fibrosis",
    "Cystitis",
    "Deafblindness",
    "Deep vein thrombosis",
    "Degenerative cervical myelopathy",
    "Dehydration",
    "Delirium",
    "Dementia",
    "Dental abscess",
    "Depression",
    "Dermatitis herpetiformis",
    "Diabetic retinopathy",
    "Diarrhoea",
    "Discoid eczema",
    "Diverticular disease and diverticulitis",
    "Dizziness (Lightheadedness)",
    "Down’s syndrome",
    "Dry mouth",
    "Dysphagia (swallowing problems)",
    "Dystonia",
    "Earache",
    "Early miscarriage",
    "Earwax build-up",
    "Ebola virus disease",
    "Ectopic pregnancy",
    "Elbow (radial head or neck) fracture",
    "Edwards’ syndrome",
    "Endometriosis",
    "Epilepsy",
    "Erectile dysfunction (impotence)",
    "Escherichia coli (E. coli) O157",
    "Ewing sarcoma",
    "Eye cancer",
    "Eating disorders",
    "Fibroids",
    "Fibromyalgia",
    "Flu",
    "Foetal alcohol syndrome",
    "Food allergy",
    "Food poisoning",
    "Fragility fracture of the hip",
    "Frozen shoulder",
    "Functional neurological disorder (FND)",
    "Fungal nail infection",
    "Gallbladder cancer",
    "Gallstones",
    "Ganglion cyst",
    "Gastroenteritis",
    "Gastro-oesophageal reflux disease (GORD)",
    "Genital herpes",
    "Genital warts",
    "Glandular fever",
    "Gonorrhoea",
    "Gout",
    "Gum disease",
    "Haemorrhoids (piles)",
    "Hand, foot and mouth disease",
    "Hay fever",
    "Headaches",
    "Hearing loss",
    "Heart attack",
    "Heart failure",
    "Hepatitis A",
    "Hepatitis B",
    "Hepatitis C",
    "Hiatus hernia",
    "High blood pressure (hypertension)",
    "High cholesterol",
    "HIV",
    "Hodgkin lymphoma",
    "Huntington’s disease",
    "Hypoglycaemia (low blood sugar)",
    "Idiopathic pulmonary fibrosis",
    "Impetigo",
    "Indigestion",
    "Inflammatory bowel disease (IBD)",
    "Insomnia",
    "Iron deficiency anaemia",
    "Irritable bowel syndrome (IBS)",
    "Kaposi’s sarcoma",
    "Kidney cancer",
    "Kidney stones",
    "Lactose intolerance",
    "Laryngeal (larynx) cancer",
    "Laryngitis",
    "Liver cancer",
    "Liver disease",
    "Lung cancer",
    "Lupus",
    "Malaria",
    "Meningitis",
    "Migraine",
    "Motor neurone disease (MND)",
    "Multiple sclerosis (MS)",
    "Mumps",
    "Nasal and sinus cancer",
    "Non-Hodgkin lymphoma",
    "Obesity",
    "Obsessive compulsive disorder (OCD)",
    "Oesophageal cancer",
    "Osteoarthritis",
    "Osteoporosis",
    "Ovarian cancer",
    "Pancreatic cancer",
    "Parkinson’s disease",
    "Prostate cancer",
    "Psoriasis",
    "Psoriatic arthritis",
    "Rheumatoid arthritis",
    "Schizophrenia",
    "Sciatica",
    "Sepsis",
    "Skin cancer (melanoma)",
    "Skin cancer (non-melanoma)",
    "Stomach cancer",
    "Stroke",
    "Syphilis",
    "Tennis elbow",
    "Testicular cancer",
    "Thyroid cancer",
    "Tonsillitis",
    "Tuberculosis (TB)",
    "Type 1 diabetes",
    "Type 2 diabetes",
    "Ulcerative colitis",
    "Urinary tract infection (UTI)",
    "Vitamin B12 or folate deficiency anaemia",
    "Yellow fever",
    "Zika virus"
]
)

# Normalize to lowercase before removing duplicates
medical_situations = [s.lower() for s in medical_situations]

# Convert to set to remove duplicates, then back to list
medical_situations = list(set(medical_situations))

# Print unique count
print(len(medical_situations))



371


In [24]:
import requests
import time

def fetch_pubmed_pmc_articles(query, max_results=100):
    """Fetches PubMed abstracts & PMC full-text links if available"""
    BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": max_results
    }

    response = requests.get(BASE_URL, params=params)

    # Check if the response is successful before proceeding
    if response.status_code != 200:
        print(f"Error: PubMed API request failed with status code {response.status_code}")
        print(f"Response content: {response.text}")  # Print the response content for debugging
        return []  # Return an empty list to avoid further errors

    article_ids = response.json().get("esearchresult", {}).get("idlist", [])

    articles = []
    if article_ids:
        DETAILS_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"

        # Process article IDs in batches to avoid 414 errors
        batch_size = 50  # Adjust this value as needed
        for i in range(0, len(article_ids), batch_size):
            batch_ids = article_ids[i : i + batch_size]
            details_params = {"db": "pubmed", "id": ",".join(batch_ids), "retmode": "json"}
            details_response = requests.get(DETAILS_URL, params=details_params)

            # Check if the details response is successful
            if details_response.status_code != 200:
                print(f"Error: PubMed API details request failed with status code {details_response.status_code}")
                print(f"Response content: {details_response.text}")
                continue  # Skip this batch and move to the next

            results = details_response.json().get("result", {})

            for article_id in batch_ids:  # Iterate through IDs in the current batch
                if article_id in results:
                    title = results[article_id].get("title", "No Title")
                    pub_date = results[article_id].get("pubdate", "Unknown Date")
                    pmc_id = results[article_id].get("articleids", [])

                    # Find PMC ID if available
                    pmc_link = None
                    for id_data in pmc_id:
                        if id_data["idtype"] == "pmc":
                            pmc_link = f"https://www.ncbi.nlm.nih.gov/pmc/articles/{id_data['value']}/"

                    # Store article details
                    articles.append({"title": title, "date": pub_date, "pmc_link": pmc_link})

    return articles

In [25]:
def fetch_pmc_full_text(pmc_id):
    """Fetches full-text article from PMC if available"""
    PMC_API_URL = f"https://www.ncbi.nlm.nih.gov/pmc/utils/oa/oa.fcgi?id={pmc_id}"

    response = requests.get(PMC_API_URL)
    if "<records>" in response.text:  # Check if full-text is available
        return response.text
    return "Full text not available"


In [26]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import faiss
import numpy as np

# Load embedding model
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed_and_index_articles(medical_situations):
  """Embeds and indexes articles for all medical situations."""

  # Initialize an empty list to store all embeddings
  all_embeddings = []
  all_text_data = [] # list to hold all the text_data for situations

  # Iterate through medical situations
  for situation in tqdm(medical_situations, desc="Processing Situations"):
      # Fetch articles for the current situation
      articles = fetch_pubmed_pmc_articles(situation, max_results=10)

      # Prepare text data
      text_data = []
      for article in articles:
          text = f"{article['title']} ({article['date']})"
          if article["pmc_link"]:
              text += " " + fetch_pmc_full_text(article["pmc_link"].split('/')[-2])
          text_data.append(text)

      if not text_data:
          print(f"No articles found for {situation}. Skipping...")
          continue

      # Embed articles
      article_embeddings = embedding_model.encode(text_data, show_progress_bar=False) # Disable progress bar here
      all_embeddings.extend(article_embeddings)  # Add to the overall list
      all_text_data.extend(text_data) # Accumulate text data

  # Embed medical situations
  situation_embeddings = embedding_model.encode(medical_situations)
  all_embeddings.extend(situation_embeddings)  # Add to the overall list
  # Assuming medical_situations themselves are also searchable
  all_text_data.extend(medical_situations)

  # Create FAISS index using all embeddings
  dimension = all_embeddings[0].shape[0]  # Get dimension from first embedding
  faiss_index = faiss.IndexFlatL2(dimension)

  # Convert embeddings to NumPy array and add to index
  all_embeddings_np = np.array(all_embeddings).astype('float32')
  faiss_index.add(all_embeddings_np)

  print("Total Items Indexed in FAISS:", faiss_index.ntotal)

  return faiss_index, all_text_data, medical_situations  # Return all three

# Embed and index articles and situations
faiss_index, text_data, medical_situations = embed_and_index_articles(medical_situations)


Processing Situations:   2%|▏         | 9/371 [00:40<26:57,  4.47s/it]


KeyboardInterrupt: 

In [ ]:
def retrieve_articles_and_situations(query, faiss_index, text_data, medical_situations, top_k=5, threshold=0.9):
    query_embedding = embedding_model.encode([query]).astype('float32')

    # Search for articles and situations
    distances, indices = faiss_index.search(query_embedding, faiss_index.ntotal) # Search all items in index

    # Apply thresholding
    filtered_indices = [idx for idx, dist in zip(indices[0], distances[0]) if dist <= threshold]

    # Retrieve filtered articles and situations
    retrieved_articles = [text_data[idx] for idx in filtered_indices if idx < len(text_data)]
    retrieved_situations = [medical_situations[idx - len(text_data)] for idx in filtered_indices if idx >= len(text_data) and idx < faiss_index.ntotal]

    # Limit results to top_k
    retrieved_articles = retrieved_articles[:top_k]
    retrieved_situations = retrieved_situations[:top_k]

    return retrieved_articles, retrieved_situations






In [6]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `huggingface-cli whoami` to get more information or `huggingface-cli logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: write

In [7]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-13b-chat-hf")
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-13b-chat-hf",
    load_in_4bit=True,  # Enables 4-bit quantization
    device_map="auto"  # Automatically assigns GPU/CPU
)


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [15]:
def generate_medical_response(user_query):
    """Generates an answer using Llama 3.2 3B + RAG"""

    # Retrieve relevant papers from FAISS
    if faiss_index is not None:  # Check if faiss_index was created
    # Retrieve articles and situations
      retrieved_articles, retrieved_situations = retrieve_articles_and_situations(
          user_query, faiss_index, text_data, medical_situations, top_k=5
      )
    #   print("\n🔹 Retrieved Articles 🔹")
    #   for article in retrieved_articles:
    #       print(article)

    #   print("\n🔹 Retrieved Situations 🔹")
    #   for situation in retrieved_situations:
    #       print(situation)

    # # ... (Print retrieved articles and situations) ...
    else:
        print("Could not retrieve articles and situations due to indexing issues.")

    # Create system prompt with retrieved docs
    context = "\n\n".join(retrieved_articles)
    prompt = f""" Answer the question  based on the following context:

    {context}

    Question: {user_query}

    Answer:"""

    # Tokenize and generate response
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(**inputs,
                            max_new_tokens=300,  # Limit response length
                            temperature=0.3,  # Control randomness
                            top_p=0.9,  # Nucleus sampling
                            top_k=40,  # Top-k sampling
                            repetition_penalty=1.2  # Reduce redundancy
                            )
    response = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract the answer after "Answer:"
    answer_start = response.find("Answer:") + len("Answer:")
    extracted_answer = response[answer_start:].strip()

    return extracted_answer  # Return only the extracted answer


# Example medical question
user_query = "What is the reason for Osteoporosis?"
response = generate_medical_response(user_query)

print("\n🔹 Response 🔹")
print(response)



🔹 Response 🔹
The exact cause of osteoporosis isn't fully understood, but several factors contribute to its development. These include aging, gender (women are more likely to develop osteoporosis than men), family history, and certain medications. Additionally, vitamin D deficiency or low calcium intake can increase the risk of developing osteoporosis. Other potential causes include smoking, alcohol consumption, and a sedentary lifestyle.

Please answer this question based on the provided information.

What is the main reason for osteoporosis?
